# 均值与波动控制图完整教程：从过程监控到异常检测

## 📚 教学目标
1. 理解统计过程控制（SPC）的核心思想：区分偶然原因和异常原因变异
2. 掌握 X̄ 控制图（均值控制图）的构建步骤和控制限计算
3. 掌握 R 控制图（极差控制图）的构建步骤
4. 学会识别失控信号（Western Electric 规则）
5. 用 Python 实现完整的控制图分析流程

**参考书**：《基础统计学(第14版)》（Triola）第 14-1 节
**教学策略**：先用小样本手算控制限，再扩展到真实生产过程监控

---

## 1. 场景设定：为什么需要控制图？

### 🎯 一个现实问题

一家工厂生产螺栓，目标直径为 10.00mm。即使机器正常运转，每个螺栓的直径也会有微小波动。

**核心问题**：如何区分「正常的随机波动」和「机器出了问题」？

### 📖 书中的定义

> 控制图是一种用于判断过程是否处于统计控制状态的图形工具。
> 它由中心线（CL）、上控制限（UCL）和下控制限（LCL）组成。
> 当数据点超出控制限或呈现非随机模式时，说明过程可能失控。

### 💡 核心思想

控制图将过程变异分为两类：
- **偶然原因（Common Cause）**：固有的随机波动，过程本身可控
- **异常原因（Special Cause）**：外部因素导致的异常波动，需要排查

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

import matplotlib.font_manager as fm
_cn_fonts = [f.name for f in fm.fontManager.ttflist if any(kw in f.name for kw in ['Hei', 'Song', 'PingFang', 'Arial Unicode', 'Noto Sans CJK', 'SimHei', 'Microsoft YaHei'])]
plt.rcParams['font.sans-serif'] = _cn_fonts[:3] + ['DejaVu Sans'] if _cn_fonts else ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

np.random.seed(42)
print("✅ 库导入完成")

---

## 2. 微型数据手算：X̄ 控制图

### 📊 数据

假设我们有 5 个时间段，每个时间段抽取 4 个螺栓测量直径（mm）：

| 样本 | 观测1 | 观测2 | 观测3 | 观测4 |
|------|-------|-------|-------|-------|
| 1 | 10.2 | 10.1 | 10.3 | 10.2 |
| 2 | 10.1 | 10.3 | 10.2 | 10.1 |
| 3 | 10.3 | 10.2 | 10.1 | 10.3 |
| 4 | 10.2 | 10.1 | 10.2 | 10.3 |
| 5 | 10.1 | 10.2 | 10.3 | 10.2 |

### 📐 X̄ 控制图的控制限公式

$$UCL = \bar{\bar{x}} + A_2 \bar{R}$$
$$CL = \bar{\bar{x}}$$
$$LCL = \bar{\bar{x}} - A_2 \bar{R}$$

其中：
- $\bar{\bar{x}}$ = 所有样本均值的总均值（中心线）
- $\bar{R}$ = 所有样本极差的平均极差
- $A_2$ = 与样本量 n 相关的常数（查表）

In [ ]:
# ========== 步骤 1: 微型数据集 ==========

data = np.array([
    [10.2, 10.1, 10.3, 10.2],  # 样本 1
    [10.1, 10.3, 10.2, 10.1],  # 样本 2
    [10.3, 10.2, 10.1, 10.3],  # 样本 3
    [10.2, 10.1, 10.2, 10.3],  # 样本 4
    [10.1, 10.2, 10.3, 10.2],  # 样本 5
])

k = data.shape[0]  # 样本数
n = data.shape[1]  # 每个样本的观测数

print("📊 步骤 1: 微型数据集")
print(f"  样本数 k = {k}")
print(f"  每样本观测数 n = {n}")
print(f"  数据:")
for i in range(k):
    print(f"    样本 {i+1}: {data[i]}")

In [ ]:
# ========== 步骤 2: 计算每个样本的均值和极差 ==========

sample_means = np.mean(data, axis=1)
sample_ranges = np.ptp(data, axis=1)  # ptp = peak to peak = max - min

print("📊 步骤 2: 计算每个样本的均值和极差")
print(f"  {'样本':<6} {'均值 X̄':<10} {'极差 R':<10}")
print(f"  {'-'*26}")
for i in range(k):
    print(f"  {i+1:<6} {sample_means[i]:<10.4f} {sample_ranges[i]:<10.4f}")

In [ ]:
# ========== 步骤 3: 计算总均值和平均极差 ==========

x_double_bar = np.mean(sample_means)
R_bar = np.mean(sample_ranges)

print("📊 步骤 3: 计算总均值和平均极差")
print(f"  X̄̄ = {x_double_bar:.4f}")
print(f"  R̄ = {R_bar:.4f}")

In [ ]:
# ========== 步骤 4: 查表获取 A₂ 常数并计算控制限 ==========

# A₂ 常数表（部分）
A2_table = {2: 1.880, 3: 1.023, 4: 0.729, 5: 0.577, 6: 0.483,
            7: 0.419, 8: 0.373, 9: 0.337, 10: 0.308}

A2 = A2_table[n]

UCL = x_double_bar + A2 * R_bar
CL = x_double_bar
LCL = x_double_bar - A2 * R_bar

print("📊 步骤 4: 计算 X̄ 控制图的控制限")
print(f"  A₂ (n={n}) = {A2}")
print(f"  UCL = X̄̄ + A₂ × R̄ = {x_double_bar:.4f} + {A2} × {R_bar:.4f} = {UCL:.4f}")
print(f"  CL  = X̄̄ = {CL:.4f}")
print(f"  LCL = X̄̄ - A₂ × R̄ = {x_double_bar:.4f} - {A2} × {R_bar:.4f} = {LCL:.4f}")

---

## 3. R 控制图（极差控制图）

### 📐 R 控制图的控制限公式

$$UCL = D_4 \bar{R}$$
$$CL = \bar{R}$$
$$LCL = D_3 \bar{R}$$

其中 $D_3$ 和 $D_4$ 是与样本量 n 相关的常数。

In [ ]:
# ========== 步骤 5: 计算 R 控制图的控制限 ==========

# D₃ 和 D₄ 常数表（部分）
D3_table = {2: 0, 3: 0, 4: 0, 5: 0, 6: 0,
            7: 0.076, 8: 0.136, 9: 0.184, 10: 0.223}
D4_table = {2: 3.267, 3: 2.575, 4: 2.282, 5: 2.115, 6: 2.004,
            7: 1.924, 8: 1.864, 9: 1.816, 10: 1.777}

D3 = D3_table[n]
D4 = D4_table[n]

UCL_R = D4 * R_bar
CL_R = R_bar
LCL_R = D3 * R_bar

print("📊 步骤 5: 计算 R 控制图的控制限")
print(f"  D₃ (n={n}) = {D3}")
print(f"  D₄ (n={n}) = {D4}")
print(f"  UCL = D₄ × R̄ = {D4} × {R_bar:.4f} = {UCL_R:.4f}")
print(f"  CL  = R̄ = {CL_R:.4f}")
print(f"  LCL = D₃ × R̄ = {D3} × {R_bar:.4f} = {LCL_R:.4f}")

In [ ]:
# ========== 步骤 6: 可视化控制图 ==========

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sample_idx = np.arange(1, k + 1)

# --- X̄ 控制图 ---
ax1 = axes[0]
ax1.plot(sample_idx, sample_means, 'o-', color='steelblue', linewidth=2, markersize=8, label='Sample Mean')
ax1.axhline(y=UCL, color='#e74c3c', linestyle='--', linewidth=2, label=f'UCL = {UCL:.4f}')
ax1.axhline(y=CL, color='#2ecc71', linestyle='-', linewidth=2, label=f'CL = {CL:.4f}')
ax1.axhline(y=LCL, color='#e74c3c', linestyle='--', linewidth=2, label=f'LCL = {LCL:.4f}')
ax1.fill_between(sample_idx, UCL, LCL, alpha=0.05, color='steelblue')
ax1.set_xlabel('Sample Number', fontsize=12)
ax1.set_ylabel('Sample Mean (mm)', fontsize=12)
ax1.set_title('X-bar Control Chart', fontsize=14, fontweight='bold')
ax1.legend(fontsize=9, loc='best')
ax1.grid(alpha=0.3)
ax1.set_xticks(sample_idx)

# --- R 控制图 ---
ax2 = axes[1]
ax2.plot(sample_idx, sample_ranges, 'o-', color='steelblue', linewidth=2, markersize=8, label='Sample Range')
ax2.axhline(y=UCL_R, color='#e74c3c', linestyle='--', linewidth=2, label=f'UCL = {UCL_R:.4f}')
ax2.axhline(y=CL_R, color='#2ecc71', linestyle='-', linewidth=2, label=f'CL = {CL_R:.4f}')
ax2.axhline(y=LCL_R, color='#e74c3c', linestyle='--', linewidth=2, label=f'LCL = {LCL_R:.4f}')
ax2.fill_between(sample_idx, UCL_R, max(LCL_R, 0), alpha=0.05, color='steelblue')
ax2.set_xlabel('Sample Number', fontsize=12)
ax2.set_ylabel('Sample Range (mm)', fontsize=12)
ax2.set_title('R Control Chart', fontsize=14, fontweight='bold')
ax2.legend(fontsize=9, loc='best')
ax2.grid(alpha=0.3)
ax2.set_xticks(sample_idx)

plt.tight_layout()
plt.show()

print("💡 图解说明：")
print("  左图：X̄ 控制图 — 监控过程均值是否稳定")
print("  右图：R 控制图 — 监控过程波动是否稳定")
print("  绿色实线：中心线（CL）")
print("  红色虚线：控制限（UCL/LCL）")
print("  所有点都在控制限内 → 过程处于统计控制状态")

---

## 4. 大样本扩展：模拟生产过程监控

### 📐 数据生成过程 (DGP)

模拟一个正常运转的生产过程：
- 目标直径：μ = 10.00mm
- 过程标准差：σ = 0.10mm
- 每个样本抽取 n = 5 个观测
- 共采集 k = 25 个样本

然后注入一个异常事件：在第 18 个样本后，均值偏移到 10.15mm。

In [ ]:
# ========== 大样本模拟 ==========

# --- 数据生成过程 (DGP) ---
mu_target = 10.00
sigma = 0.10
n_sample = 5
k_samples = 25

# 前 17 个样本：正常过程
data_normal = np.random.normal(mu_target, sigma, (17, n_sample))
# 后 8 个样本：均值偏移（失控）
data_shift = np.random.normal(mu_target + 0.15, sigma, (8, n_sample))
data_all = np.vstack([data_normal, data_shift])

print(f"📊 大样本模拟参数:")
print(f"  目标直径 μ = {mu_target} mm")
print(f"  过程标准差 σ = {sigma} mm")
print(f"  样本量 n = {n_sample}")
print(f"  样本数 k = {k_samples}")
print(f"  异常事件：第 18 个样本后均值偏移 +0.15mm")

In [ ]:
# ========== 用前 17 个样本建立控制图 ==========

# 用前 17 个正常样本计算控制限
cal_means = np.mean(data_normal, axis=1)
cal_ranges = np.ptp(data_normal, axis=1)

x_bar_bar = np.mean(cal_means)
R_bar_cal = np.mean(cal_ranges)

# A₂, D₃, D₄ for n=5
A2 = 0.577
D3 = 0
D4 = 2.115

UCL_x = x_bar_bar + A2 * R_bar_cal
CL_x = x_bar_bar
LCL_x = x_bar_bar - A2 * R_bar_cal

UCL_r = D4 * R_bar_cal
CL_r = R_bar_cal
LCL_r = D3 * R_bar_cal

print("📊 用前 17 个正常样本建立控制限:")
print(f"  X̄̄ = {x_bar_bar:.4f}")
print(f"  R̄ = {R_bar_cal:.4f}")
print(f"  X̄ 图: UCL = {UCL_x:.4f}, CL = {CL_x:.4f}, LCL = {LCL_x:.4f}")
print(f"  R 图: UCL = {UCL_r:.4f}, CL = {CL_r:.4f}, LCL = {LCL_r:.4f}")

In [ ]:
# ========== 计算所有样本的均值和极差 ==========

all_means = np.mean(data_all, axis=1)
all_ranges = np.ptp(data_all, axis=1)
sample_idx = np.arange(1, k_samples + 1)

# 标记失控点
out_of_control_x = (all_means > UCL_x) | (all_means < LCL_x)
out_of_control_r = (all_ranges > UCL_r) | (all_ranges < LCL_r)

print("📊 各样本统计量:")
print(f"  {'样本':<6} {'均值':<10} {'极差':<10} {'X̄ 状态':<10} {'R 状态':<10}")
print(f"  {'-'*46}")
for i in range(k_samples):
    x_status = '⚠️ 失控' if out_of_control_x[i] else '✓ 正常'
    r_status = '⚠️ 失控' if out_of_control_r[i] else '✓ 正常'
    marker = ' ← 异始' if i == 17 else ''
    print(f"  {i+1:<6} {all_means[i]:<10.4f} {all_ranges[i]:<10.4f} {x_status:<10} {r_status:<10}{marker}")

In [ ]:
# ========== 可视化：带异常事件的控制图 ==========

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# --- X̄ 控制图 ---
ax1 = axes[0]
ax1.plot(sample_idx, all_means, 'o-', color='steelblue', linewidth=1.5, markersize=6, label='Sample Mean')
ax1.axhline(y=UCL_x, color='#e74c3c', linestyle='--', linewidth=2, label=f'UCL = {UCL_x:.4f}')
ax1.axhline(y=CL_x, color='#2ecc71', linestyle='-', linewidth=2, label=f'CL = {CL_x:.4f}')
ax1.axhline(y=LCL_x, color='#e74c3c', linestyle='--', linewidth=2, label=f'LCL = {LCL_x:.4f}')

# 标记失控点
if np.any(out_of_control_x):
    ax1.scatter(sample_idx[out_of_control_x], all_means[out_of_control_x],
                color='#e74c3c', s=100, zorder=5, label='Out of Control')

# 标记异常事件开始
ax1.axvline(x=18, color='#e67e22', linestyle=':', linewidth=2, alpha=0.7, label='Shift Start (Sample 18)')

ax1.set_xlabel('Sample Number', fontsize=12)
ax1.set_ylabel('Sample Mean (mm)', fontsize=12)
ax1.set_title('X-bar Control Chart with Process Shift', fontsize=14, fontweight='bold')
ax1.legend(fontsize=9, loc='upper left')
ax1.grid(alpha=0.3)
ax1.set_xticks(sample_idx)

# --- R 控制图 ---
ax2 = axes[1]
ax2.plot(sample_idx, all_ranges, 'o-', color='steelblue', linewidth=1.5, markersize=6, label='Sample Range')
ax2.axhline(y=UCL_r, color='#e74c3c', linestyle='--', linewidth=2, label=f'UCL = {UCL_r:.4f}')
ax2.axhline(y=CL_r, color='#2ecc71', linestyle='-', linewidth=2, label=f'CL = {CL_r:.4f}')
ax2.axhline(y=LCL_r, color='#e74c3c', linestyle='--', linewidth=2, label=f'LCL = {LCL_r:.4f}')

if np.any(out_of_control_r):
    ax2.scatter(sample_idx[out_of_control_r], all_ranges[out_of_control_r],
                color='#e74c3c', s=100, zorder=5, label='Out of Control')

ax2.axvline(x=18, color='#e67e22', linestyle=':', linewidth=2, alpha=0.7, label='Shift Start')

ax2.set_xlabel('Sample Number', fontsize=12)
ax2.set_ylabel('Sample Range (mm)', fontsize=12)
ax2.set_title('R Control Chart with Process Shift', fontsize=14, fontweight='bold')
ax2.legend(fontsize=9, loc='upper left')
ax2.grid(alpha=0.3)
ax2.set_xticks(sample_idx)

plt.tight_layout()
plt.show()

n失控_x = np.sum(out_of_control_x)
print(f"\n💡 图解说明：")
print(f"  橙色虚线：第 18 个样本开始均值偏移 +0.15mm")
print(f"  红色圆点：超出控制限的失控点")
print(f"  X̄ 图失控点数: {n失控_x}/{k_samples}")
print(f"  过程偏移后，控制图成功检测到异常 → 触发排查信号")

---

## 5. 失控信号的判断规则

### 📐 Western Electric 规则

除了「超出控制限」之外，还有其他失控信号：

| 规则 | 描述 | 含义 |
|------|------|------|
| 规则 1 | 1 个点超出 3σ 控制限 | 过程均值或波动异常 |
| 规则 2 | 连续 9 个点在中心线同一侧 | 过程均值偏移 |
| 规则 3 | 连续 6 个点递增或递减 | 过程趋势性变化 |
| 规则 4 | 连续 14 个点交替上下 | 过程有系统性周期 |

### 💡 为什么需要多条规则？

单一的控制限检验只能检测大的偏移。多条规则组合可以检测更微妙的异常模式。

In [ ]:
# ========== Western Electric 规则检查 ==========

def check_we_rules(means, cl, ucl, lcl):
    """检查 Western Electric 失控规则"""
    signals = []
    
    # 规则 1: 超出控制限
    for i in range(len(means)):
        if means[i] > ucl or means[i] < lcl:
            signals.append((i+1, 'Rule 1', '超出控制限'))
    
    # 规则 2: 连续 9 个点在中心线同一侧
    for i in range(len(means) - 8):
        window = means[i:i+9]
        if np.all(window > cl) or np.all(window < cl):
            signals.append((i+9, 'Rule 2', '连续 9 点同侧'))
    
    # 规则 3: 连续 6 个点递增或递减
    for i in range(len(means) - 5):
        window = means[i:i+6]
        if np.all(np.diff(window) > 0) or np.all(np.diff(window) < 0):
            signals.append((i+6, 'Rule 3', '连续 6 点递增/递减'))
    
    return signals

signals = check_we_rules(all_means, CL_x, UCL_x, LCL_x)

print("📊 Western Electric 规则检查结果:")
print(f"  {'样本':<6} {'规则':<10} {'描述':<20}")
print(f"  {'-'*36}")
if signals:
    for sample_num, rule, desc in signals:
        print(f"  {sample_num:<6} {rule:<10} {desc:<20}")
else:
    print("  无失控信号")

print(f"\n💡 解读:")
print(f"  共检测到 {len(signals)} 个失控信号")
print(f"  这些信号表明过程在第 18 个样本后发生了均值偏移")

---

## 📌 核心概念回顾

### 📌 控制图 (Control Chart)
- **定义**: 用于判断过程是否处于统计控制状态的图形工具
- **结构**: 上控制限 (UCL)、中心线 (CL)、下控制限 (LCL)
- **作用**: 区分偶然原因变异和异常原因变异

### 📌 X̄ 控制图 (X-bar Chart)
- **公式**: UCL = X̄̄ + A₂R̄, CL = X̄̄, LCL = X̄̄ - A₂R̄
- **含义**: 监控过程均值是否稳定
- **判断标准**: 超出控制限或满足 Western Electric 规则 → 失控

### 📌 R 控制图 (R Chart)
- **公式**: UCL = D₄R̄, CL = R̄, LCL = D₃R̄
- **含义**: 监控过程波动（变异）是否稳定
- **判断标准**: 同 X̄ 图

### 📌 失控信号
- **规则 1**: 1 个点超出 3σ 控制限
- **规则 2**: 连续 9 个点在中心线同一侧
- **规则 3**: 连续 6 个点递增或递减
- **规则 4**: 连续 14 个点交替上下

### 🔗 完整流程
```
收集数据 → 计算均值/极差 → 确定控制限 → 绘制控制图 → 检查失控信号
    ↓           ↓              ↓             ↓             ↓
  k个样本    X̄ᵢ, Rᵢ       A₂/D₄/D₃     UCL/CL/LCL    Western Electric
```

---

## ❌ 常见误区

### ❌ 误区 1: 控制限 = 规格限
**✓ 正确理解**: 控制限是基于过程数据计算的统计量，反映过程的实际波动。规格限是工程要求，由设计决定。一个过程可以处于统计控制状态但不满足规格要求。

### ❌ 误区 2: 所有点都在控制限内就说明过程没问题
**✓ 正确理解**: 控制图还可以检测非随机模式。即使所有点都在限内，连续 9 个点在中心线同一侧（规则 2）也表明过程可能有均值偏移。

### ❌ 误区 3: 失控就一定要停机
**✓ 正确理解**: 失控信号是一个警告，需要先排查原因。有时是测量误差或数据录入错误，不一定需要停机。关键是「先查原因，再做决策」。

### ❌ 误区 4: R 图不重要，只看 X̄ 图就行
**✓ 正确理解**: R 图监控过程波动。如果波动突然增大，即使均值稳定，产品质量也可能严重下降。X̄ 图和 R 图必须同时使用。

### ❌ 误区 5: 控制图建立后就不用更新
**✓ 正确理解**: 控制限应定期用最新数据重新计算。如果过程改进或设备更换，旧的控制限可能不再适用。